# Hands-on: LLM-as-a-Judge für Softwarequalität

**Hinweis:** Das Setup lädt den Code automatisch. Klappt das nicht, ladet die ZIP [hier von Google Drive](https://drive.google.com/file/d/192TGNh2VbD1e-Qp_WVU7CJn5Ra31uSAs/view?usp=sharing) herunter, ladet sie in Colab links über 📁 Dateien → Hochladen hoch (Name `software-quality-engineering.zip` beibehalten) und führt das Setup noch einmal aus.

In [ ]:
#@title ⚙️ Setup (einmal ausführen)

import os
import subprocess
import sys

REPO_URL = "https://github.com/evabowe/software-quality-engineering"
# Ersatz, solange das GitHub-Repo nicht öffentlich ist: ZIP des Repos auf Google Drive.
ZIP_URL = "https://drive.usercontent.google.com/download?id=192TGNh2VbD1e-Qp_WVU7CJn5Ra31uSAs&export=download&confirm=t"
# Lokal (Jupyter im Repo) wird der vorhandene Checkout genutzt, in Colab ein frischer Klon.
if os.path.isfile("../pyproject.toml"):
    REPO_DIR = os.path.abspath("..")
else:
    REPO_DIR = "/content/software-quality-engineering"
    ZIP_PATH = "/content/software-quality-engineering.zip"
    if not os.path.isdir(REPO_DIR):
        geklont = subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, REPO_DIR],
                                 capture_output=True).returncode == 0
        if not geklont:
            # GitHub nicht erreichbar: von Hand hochgeladene ZIP nehmen, sonst die von Google Drive laden.
            import urllib.request
            import zipfile

            if not os.path.isfile(ZIP_PATH):
                urllib.request.urlretrieve(ZIP_URL, ZIP_PATH)
            with zipfile.ZipFile(ZIP_PATH) as archiv:
                archiv.extractall("/content")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR], check=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)  # die Installation wird sonst erst nach einem Neustart sichtbar

import html

import ipywidgets as widgets
import pandas as pd
from IPython.display import HTML, display

from judge_lab import bewerte, bewerte_alle, sende, verbinde
from judge_lab.snippets import get_snippet, load_snippets

# --- Hilfen für die Zellen der Runden ---------------------------------------------

SNIPPETS = load_snippets()
TITEL = {s.id: s.title for s in SNIPPETS}


def prompt_feld(platzhalter, hoehe="12em"):
    feld = widgets.Textarea(placeholder=platzhalter, layout=widgets.Layout(width="100%", height=hoehe))
    feld.add_class("judge-prompt")
    return feld


def ohne_rechtschreibpruefung():
    # Textarea hat keine Option dafür: Rechtschreibprüfung des Browsers (Englisch) in den Feldern abschalten.
    display(HTML("""<script>
new MutationObserver(() => document.querySelectorAll(".judge-prompt textarea")
    .forEach(t => t.spellcheck = false)).observe(document.body, {childList: true, subtree: true});
</script>"""))


def snippet_auswahl(beschreibung="Code zum Reviewen:"):
    optionen = [(f"{s.title} ({s.level or 'ohne Stufe'})", s.id) for s in SNIPPETS]
    return widgets.Dropdown(options=optionen, value="bank_account", description=beschreibung,
                            style={"description_width": "initial"})


def snippet_html(snippet_id):
    snippet = get_snippet(snippet_id)
    return (f"<h4>{html.escape(snippet.title)} <small>({snippet.level or 'ohne Stufe'})</small></h4>"
            f"<pre style='font-size:13px;line-height:1.35'>{html.escape(snippet.numbered_code())}</pre>")


def ergebnis_html(ergebnis, titel=""):
    kopf = f"<b>{html.escape(titel)}</b><br>" if titel else ""
    if ergebnis["fehler"]:
        return f"{kopf}<span style='color:#b45309'>⚠️ {html.escape(ergebnis['fehler'])} Bitte noch einmal bewerten.</span>"
    return (f"{kopf}🎯 Judge-Score <b>{ergebnis['judge_score']}</b> von max. {ergebnis['judge_max']} · "
            f"📝 Code-Score <b>{ergebnis['code_score']:.1f}</b> von 10<br>"
            f"✔ {ergebnis['gefunden']} von {ergebnis['fehler_gesamt']} echten Fehlern gefunden · "
            f"✘ {ergebnis['fehlalarme']} Fehlalarm(e)")


def ergebnis_tabelle(ergebnisse):
    zeilen = {}
    for e in ergebnisse:
        ok = not e["fehler"]
        zeilen[TITEL.get(e["snippet"], e["snippet"])] = {
            "Stufe": e["komplexitaet"],
            "🎯 Judge-Score": f"{e['judge_score']} / {e['judge_max']}" if ok else "–",
            "📝 Code-Score": f"{e['code_score']:.1f}" if ok else "–",
            "✔ gefunden": f"{e['gefunden']} / {e['fehler_gesamt']}" if ok else "–",
            "✘ Fehlalarme": e["fehlalarme"] if ok else "–",
            "Hinweis": e["fehler"] or "",
        }
    return pd.DataFrame.from_dict(zeilen, orient="index")


def hinweis_fehlgeschlagen(ergebnisse):
    if any(e["fehler"] for e in ergebnisse):
        print("⚠️ Nicht alle Stufen haben geklappt. Fehlgeschlagene Stufen werden nicht gesendet – "
              "da Senden nur einmal geht, lieber vorher noch einmal bewerten.")


def summen_tabelle(ergebnisse):
    tabelle = ergebnis_tabelle(ergebnisse)
    ok = [e for e in ergebnisse if not e["fehler"]]
    if ok:
        tabelle.loc["Summe"] = [
            "",
            f"{sum(e['judge_score'] for e in ok)} / {sum(e['judge_max'] for e in ok)}",
            f"Ø {sum(e['code_score'] for e in ok) / len(ok):.1f}",
            f"{sum(e['gefunden'] for e in ok)} / {sum(e['fehler_gesamt'] for e in ok)}",
            sum(e["fehlalarme"] for e in ok),
            "",
        ]
    return tabelle


WARTEN_HTML = """<style>@keyframes judge-dreh {to {transform: rotate(360deg)}}</style>
<div style="display:flex;align-items:center;gap:.6em;padding:.4em 0;color:#555">
  <div style="width:1.1em;height:1.1em;border:3px solid #ccc;border-top-color:#1a73e8;border-radius:50%;
              animation:judge-dreh .8s linear infinite"></div>
  <span>⏳ {text}</span>
</div>"""


def knopf(beschreibung, ausgabe, aktion, stil="primary", aktiv=True, aktiv_danach=lambda: True,
          warten="Bitte warten …"):
    """Button, der während der Aktion gesperrt ist; Meldungen der Funktionen erscheinen in ``ausgabe``.

    Solange die Aktion läuft, steht ``warten`` mit einem drehenden Kreis in der Ausgabe.
    """
    button = widgets.Button(description=beschreibung, button_style=stil, disabled=not aktiv,
                            layout=widgets.Layout(width="auto"))

    def klick(_):
        button.disabled = True
        ausgabe.clear_output()
        hinweis = widgets.HTML(WARTEN_HTML.replace("{text}", html.escape(warten)))
        try:
            with ausgabe:
                display(hinweis)
                aktion()
        except Exception as exc:  # noqa: BLE001 – im Workshop lieber Meldung als Traceback
            with ausgabe:
                display(HTML(f"<b>⚠️ Unerwarteter Fehler ({html.escape(type(exc).__name__)}). Bitte Bescheid geben.</b>"))
        finally:
            hinweis.close()
            button.disabled = not aktiv_danach()

    button.on_click(klick)
    return button


# Runden, aus denen schon gesendet wurde. Bleibt erhalten, wenn eine Zelle (auch das Setup)
# erneut ausgeführt wird; erst ein Neustart der Laufzeit setzt es zurück.
GESENDET = globals().get("GESENDET", set())
GESENDETES = globals().get("GESENDETES", {})  # Runde -> gesendetes Ergebnis (Runde 3 nutzt den Prompt aus Runde 2)


def senden_knopf(ausgabe, runde, ergebnis, anzeige=None):
    """„Ans Dashboard senden“ – pro Runde nur einmal möglich, danach gesperrt.

    ``ergebnis()`` liefert, was gesendet wird; ``anzeige()`` zeigt es vorher noch einmal an.
    Gibt den Button und ``freigeben(ok)`` zurück, das die Bewerten-Funktion nach jeder Bewertung aufruft.
    """
    def senden():
        if runde in GESENDET:
            print("🔒 Ihr habt in dieser Runde schon gesendet – senden geht nur einmal.")
            return
        if anzeige:
            anzeige()
        if sende(ergebnis(), runde=runde):
            GESENDET.add(runde)
            GESENDETES[runde] = ergebnis()
            button.description = "🔒 Gesendet"

    button = knopf("Ans Dashboard senden (nur 1×)", ausgabe, senden, stil="success", aktiv=False,
                   warten="Wird gesendet …",
                   aktiv_danach=lambda: runde not in GESENDET)
    if runde in GESENDET:
        button.description = "🔒 Gesendet"

    def freigeben(ok):
        button.disabled = runde in GESENDET or not ok

    return button, freigeben


# --- Verbinden ----------------------------------------------------------------------

gruppencode = widgets.Text(description="Gruppencode:", placeholder="von der Folie",
                           style={"description_width": "initial"})
verbinden_ausgabe = widgets.Output()
verbinden_knopf = knopf("Verbinden", verbinden_ausgabe, lambda: verbinde(gruppencode.value),
                       warten="Verbinde …")
display(widgets.HBox([gruppencode, verbinden_knopf]), verbinden_ausgabe)

## Schritt 0: Die Code-Beispiele ansehen

Bevor ihr einen Judge-Prompt schreibt, schaut euch die sechs Code-Beispiele selbst genau an. **Führt dafür die Zelle „🔍 Code ansehen“ darunter aus** und wählt im Dropdown ein Beispiel.

Untersucht jedes Beispiel im Paar auf:

1. **Semantik:** Was soll der Code tun, und tut er das auch?
2. **Geschätzte Code-Komplexität:** Wie schwer ist der Code zu verstehen – eher einfach, mittel oder schwer?
3. **Fehler:** Fallen euch direkt schon Fehler auf?
4. **Eure eigene Bewertung:** Wie würdet ihr den Code beim Überfliegen bewerten (z. B. als Schulnote)?

**Falls kein Dropdown erscheint:** Die Beispiele liegen auch als Dateien im Ordner `snippets/`. In Colab links auf **📁 Dateien** klicken, dann `software-quality-engineering` → `snippets`

| Beispiel | Datei |
|---|---|
| Bankkonto | `snippets/bank_account/bank_account.py` |
| Bitcount | `snippets/qb_bitcount/bitcount.py` |
| Levenshtein Distance | `snippets/qb_levenshtein/levenshtein.py` |
| Next Permutation | `snippets/qb_next_permutation/next_permutation.py` |
| QuickSort | `snippets/qb_quicksort/quicksort.py` |
| Infix to RPN Conversion | `snippets/qb_shunting_yard/shunting_yard.py` |

**Wichtig:** Schaut euch in den Ordnern nur die Code-Dateien aus der Tabelle an – nicht die anderen Dateien (`meta.yaml`, `SOLUTION.md`, `tests.py`). Darin stehen die Lösungen bzw. die bekannten Fehler.

In [ ]:
#@title 🔍 Code ansehen
code_auswahl = snippet_auswahl("Code ansehen:")
code_ausgabe = widgets.Output()


def code_zeigen(*_):
    code_ausgabe.clear_output()
    with code_ausgabe:
        display(HTML(snippet_html(code_auswahl.value)))


code_auswahl.observe(code_zeigen, names="value")
code_zeigen()
display(code_auswahl, code_ausgabe)

## Runde 1 – Frei ausprobieren: Was gehört in einen Judge-Prompt?

Habt ihr schon mal selber einen Judge-Prompt, bzw. eine Review-Anweisung an ein LLM verfasst? Was sollte in solchen Prompts allgemein alles drinstehen, um möglichst alle Fehler im Code zu finden? **Verfasst folgend einen möglichen Prompt für einen LLM-Judge.**

**So geht's:**

1. **Führt die Zelle „✍️ Runde 1“ darunter aus (▶).** Dann erscheint das Textfeld für euren Prompt
2. Schreibt euren Prompt ins Textfeld und wählt darunter bei **Code zum Reviewen** ein Beispiel.
3. Klickt auf **Bewerten** – erst jetzt bewertet der Judge den Code. Bewerten könnt ihr **beliebig oft**: Prompt ändern, neu bewerten, vergleichen.
4. Euer bestes Ergebnis schickt ihr mit **Ans Dashboard senden** an den Beamer. **Senden geht pro Runde nur einmal** – danach ist der Button gesperrt. Gesendet wird die zuletzt angezeigte Bewertung.

- 🎯 **Judge-Score:** Wie gut war der Judge? 2 Punkte für jeden echten Fehler, den er findet, −1 für jeden Fehlalarm.
- 📝 **Code-Score:** Welche Note gibt der Judge dem Code (0–10)?

In [ ]:
#@title ✍️ Runde 1: Euren Prompt bewerten
r1_prompt = prompt_feld("Schreibe einen Judge-Prompt …", hoehe="15em")
r1_snippet = snippet_auswahl()
r1_ausgabe = widgets.Output()
r1_ergebnis = {}


def r1_bewerten():
    r1_ergebnis["letztes"] = bewerte(r1_prompt.value, snippet=r1_snippet.value)
    display(HTML(ergebnis_html(r1_ergebnis["letztes"])))
    r1_freigeben(not r1_ergebnis["letztes"]["fehler"])


r1_bewerten_knopf = knopf("Bewerten", r1_ausgabe, r1_bewerten, warten="Der Judge ist am Arbeiten …")
r1_senden, r1_freigeben = senden_knopf(r1_ausgabe, 1, lambda: r1_ergebnis["letztes"],
                                       lambda: display(HTML(ergebnis_html(r1_ergebnis["letztes"], "Gesendet wird:"))))
display(r1_prompt, widgets.HBox([r1_snippet, r1_bewerten_knopf, r1_senden]), r1_ausgabe)
ohne_rechtschreibpruefung()


## Runde 2 – Rollen vergleichen

Formuliert selbst zwei **möglichst gegensätzliche Rollen** für euren Judge – z. B. eine strenge und eine wohlwollende, oder eine Sicherheitsexpertin und einen Product Owner unter Zeitdruck. Je stärker sich die Rollen unterscheiden, desto besser seht ihr, was die Rolle am Ergebnis ändert.

- Wählt oben **ein** Beispiel; beide Rollen werden auf demselben Code bewertet.
- Ans Dashboard geht **nur eine** der beiden Rollen: Wählt sie bei **Senden** aus.

**Leitfragen (im Paar besprechen)**

1. Wie unterscheidet sich der **Code-Score** zwischen euren Rollen?
2. Ändert die Rolle auch, **wie viele echte Fehler gefunden** werden – oder nur die Note?

In [ ]:
#@title 🎭 Runde 2: Zwei Rollen bewerten
r2_rolle_a = prompt_feld("Rolle A: Beschreibt eine Rolle für den Judge …")
r2_rolle_b = prompt_feld("Rolle B: Beschreibt eine möglichst gegensätzliche Rolle …")
r2_snippet = snippet_auswahl()
r2_ausgabe_a, r2_ausgabe_b, r2_ausgabe = widgets.Output(), widgets.Output(), widgets.Output()
r2_ergebnis = {}


def r2_bewerten(rolle, feld, anderer_knopf):
    # Solange eine Rolle bewertet wird, ist der Bewerten-Button der anderen Rolle gesperrt.
    anderer_knopf.disabled = True
    try:
        r2_ergebnis[rolle] = bewerte(feld.value, snippet=r2_snippet.value)
    finally:
        anderer_knopf.disabled = False
    display(HTML(ergebnis_html(r2_ergebnis[rolle], f"Rolle {rolle.upper()}")))
    r2_freigeben_auswahl()


def r2_freigeben_auswahl(*_):
    gewaehlt = r2_ergebnis.get(r2_wahl.value)
    r2_freigeben(gewaehlt is not None and not gewaehlt["fehler"])


r2_bewerten_a = knopf("Rolle A bewerten", r2_ausgabe_a, lambda: r2_bewerten("a", r2_rolle_a, r2_bewerten_b),
                      warten="Der Judge ist am Arbeiten …")
r2_bewerten_b = knopf("Rolle B bewerten", r2_ausgabe_b, lambda: r2_bewerten("b", r2_rolle_b, r2_bewerten_a),
                      warten="Der Judge ist am Arbeiten …")
r2_wahl = widgets.ToggleButtons(options=[("Rolle A", "a"), ("Rolle B", "b")], description="Senden:",
                                style={"description_width": "initial"})
r2_senden, r2_freigeben = senden_knopf(
    r2_ausgabe, 2, lambda: r2_ergebnis[r2_wahl.value],
    lambda: display(HTML(ergebnis_html(r2_ergebnis[r2_wahl.value], f"Gesendet wird Rolle {r2_wahl.value.upper()}:"))))
r2_wahl.observe(r2_freigeben_auswahl, names="value")
spalte = widgets.Layout(width="50%")
display(r2_snippet,
        widgets.HBox([widgets.VBox([r2_rolle_a, r2_bewerten_a, r2_ausgabe_a], layout=spalte),
                      widgets.VBox([r2_rolle_b, r2_bewerten_b, r2_ausgabe_b], layout=spalte)]),
        widgets.HBox([r2_wahl, r2_senden]), r2_ausgabe)
ohne_rechtschreibpruefung()


## Runde 3 – Komplexität vergleichen

Jetzt läuft **der Prompt, den ihr in Runde 2 gesendet habt**, über alle fünf Beispiele mit Komplexitätsstufe, von **L1** (einfach) bis **L5** (schwer).

Ihr seht danach die Scores **aller Stufen**. Ans Dashboard geht automatisch **nur das beste Ergebnis** (höchster Judge-Score, ⭐ in der Tabelle).

**Leitfragen (im Paar besprechen)**

1. Findet der Judge bei komplexerem Code **weniger echte Fehler**?
2. Passt die Stufe zu eurer eigenen Einschätzung aus Schritt 0?

In [ ]:
#@title 📈 Runde 3: Prompt aus Runde 2 über alle Stufen L1–L5
r3_ausgabe = widgets.Output()
r3_ergebnis = {}


def r3_laufen():
    if 3 in GESENDET:
        print("🔒 Ihr habt in dieser Runde schon gesendet – senden geht nur einmal.")
        return
    if 2 not in GESENDETES:
        print("❌ Sendet zuerst in Runde 2 eine Rolle ans Dashboard – ihr Prompt wird hier übernommen.")
        return
    prompt = GESENDETES[2]["prompt"]
    display(HTML(f"<b>Prompt aus Runde 2:</b><pre style='white-space:pre-wrap'>{html.escape(prompt)}</pre>"))
    liste = r3_ergebnis["liste"] = bewerte_alle(prompt)
    ok = [e for e in liste if not e["fehler"]]
    tabelle = ergebnis_tabelle(liste)
    if not ok:
        display(tabelle)
        print("⚠️ Keine Stufe hat geklappt – nichts gesendet. Bitte noch einmal laufen lassen.")
        return
    bestes = max(ok, key=lambda e: e["judge_score"])  # bei Gleichstand die niedrigste Stufe
    tabelle["Senden"] = ["⭐ bestes – wird gesendet" if e is bestes else "" for e in liste]
    display(tabelle)
    if len(ok) < len(liste):
        print("⚠️ Nicht alle Stufen haben geklappt – das beste Ergebnis wurde nur unter den erfolgreichen Stufen gewählt.")
    if sende([bestes], runde=3):
        GESENDET.add(3)
        GESENDETES[3] = bestes
        r3_knopf.description = "🔒 Gesendet"


r3_knopf = knopf("🔒 Gesendet" if 3 in GESENDET else "Alle Stufen bewerten und bestes Ergebnis senden",
                 r3_ausgabe, r3_laufen, stil="success", aktiv=3 not in GESENDET,
                 aktiv_danach=lambda: 3 not in GESENDET,
                 warten="Der Judge ist am Arbeiten …")
display(r3_knopf, r3_ausgabe)


## Runde 4 – AI Challenge

**Auftrag:** Findet den besten Prompt, um exakt die vorhandenen Fehler zu finden – **möglichst viele echte Fehler bei möglichst wenigen Fehlalarmen** (2 Punkte pro Treffer, −1 pro Fehlalarm). Am Dashboard wird die beste Gruppe hervorgehoben.

**Regeln**

- **Bewerten** ist unbegrenzt. Es läuft immer über alle fünf Stufen L1–L5 – euer Prompt muss also allgemein funktionieren. Bewertungen werden nicht gesendet.
- **Einreichen** geht **nur einmal**: Damit geht eure zuletzt angezeigte Bewertung ans Dashboard, danach ist der Button gesperrt. Reicht also erst ein, wenn ihr mit dem Ergebnis zufrieden seid.

**Tipps**

1. **Rolle:** Gebt dem Judge eine Rolle, die *gründlich* statt *streng* ist.
2. **Prüfschritte:** Lasst ihn systematisch vorgehen, z. B. jede Bedingung, jede Schleife und jeden Rückgabewert prüfen.
3. **Randfälle:** Leere Eingaben, Duplikate, Grenzwerte, Terminierung.
4. **Spezifikation zuerst:** Erst lesen, was der Code laut Docstring tun soll, dann den Code dagegen prüfen.
5. **Nur sichere Befunde:** Fehlalarme kosten Punkte – lieber weniger, dafür begründete Befunde.

In [ ]:
#@title 🏆 Runde 4: Bewerten und Einreichen
r4_prompt = prompt_feld("Euer Prompt für die Challenge …", hoehe="18em")
r4_ausgabe = widgets.Output()
r4_ergebnis = {}


def r4_bewerten():
    r4_ergebnis["liste"] = bewerte_alle(r4_prompt.value)
    display(summen_tabelle(r4_ergebnis["liste"]))
    hinweis_fehlgeschlagen(r4_ergebnis["liste"])
    r4_freigeben(any(not e["fehler"] for e in r4_ergebnis["liste"]))


r4_bewerten_knopf = knopf("Alle Stufen bewerten", r4_ausgabe, r4_bewerten, warten="Der Judge ist am Arbeiten …")
r4_senden, r4_freigeben = senden_knopf(r4_ausgabe, 4, lambda: r4_ergebnis["liste"],
                                       lambda: display(summen_tabelle(r4_ergebnis["liste"])))
r4_senden.description = "🔒 Eingereicht" if 4 in GESENDET else "Einreichen (nur 1×)"
display(r4_prompt, widgets.HBox([r4_bewerten_knopf, r4_senden]), r4_ausgabe)
ohne_rechtschreibpruefung()


## 🎉 Abschluss

Danke fürs Mitmachen! Bitte beantwortet zum Schluss noch einmal die Umfrage vom Anfang (Link auf der Folie):

*„Würdet ihr einem LLM das Code-Review überlassen – würdet ihr einen LLM-Judge verwenden?“*

Hat sich eure Einschätzung durch den Vortrag und das Hands-on verändert? Wenn ja, wodurch?